# 03 Generate Candidate Pairs
This notebook converts the cleaned company records into candidate pairs for entity resolution, keeping the input company fields, candidate company fields, location fields, search text, and website domain needed for scoring.

In [0]:
# Build candidate pairs from cleaned input and candidate company records.
from pyspark.sql import functions as F

source_table = "workspace.entity_resolution_project.company_er_clean"
target_table = "workspace.entity_resolution_project.company_er_candidates"

df = spark.table(source_table)

candidates = (
    df
    .filter(F.col("right_clean_company_name").isNotNull())
    .select(
        F.col("input_row_key").alias("left_row_key"),
        F.col("unique_id").alias("right_row_key"),

        F.col("input_row_key").alias("left_input_row_key"),
        F.col("unique_id").alias("candidate_unique_id"),

        F.col("input_company_name").alias("left_company_name"),
        F.col("company_name").alias("right_company_name"),

        F.col("left_clean_company_name"),
        F.col("right_clean_company_name"),

        F.col("left_country_code"),
        F.col("left_country"),

        F.col("right_country_code"),
        F.col("right_country"),

        F.col("left_city"),
        F.col("right_city"),

        F.col("left_search_text"),
        F.col("right_search_text"),

        F.col("website_domain").alias("right_website_domain")
    )
)

(
    candidates.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("Candidate rows:", candidates.count())
print("Input records:", candidates.select("left_row_key").distinct().count())

display(candidates)

In [0]:
# Check the generated candidate table schema.
candidates_check = spark.table("workspace.entity_resolution_project.company_er_candidates")
print(candidates_check.columns)